# Import

In [6]:
from datetime import datetime, timedelta
import time
from pathlib import Path
import sys
import pandas as pd
import re
from typing import List, Optional, Union, Any, Dict
import os
import finnhub
from dotenv import load_dotenv
import yfinance as yf

# CONSTANTS

In [7]:
# ==========================================
# CONFIGURATION & CONSTANTS
# ==========================================
# Options: "all.csv", "sp500.csv", "top_100.csv", "top_200.csv", "top_50.csv"
TICKER_SET: str = "all.csv"
INDUSTRIES_FILTER: Optional[List[str]] = ["Indust", "tech", "finance"]
MIN_PRICE_FILTER: Optional[float] = 400.0
MAX_PRICE_FILTER: Optional[float] = 500.0
MIN_MARKET_CAP_FILTER: Optional[float] = None
MAX_MARKET_CAP_FILTER: Optional[float] = None


# Functions

In [8]:
def get_latest_ticker_file(ticker_set: str, folder_path: Path) -> Path | None:
    """Finds the file with the most recent DD-MM-YY date prefix for the given ticker_set."""
    file_prefix = Path(ticker_set).stem  # e.g., 'top_50'

    # Search pattern: *_{file_prefix}_ticker_data.csv (e.g., *_top_50_ticker_data.csv)
    pattern = f"*_{file_prefix}_ticker_data.csv"
    matching_files = list(folder_path.glob(pattern))

    if not matching_files:
        return None

    dated_files = []
    for file in matching_files:
        # File format: DD-MM-YY_{prefix}_ticker_data.csv -> extract DD-MM-YY
        date_part = file.name.split("_")[0]
        try:
            file_date = datetime.strptime(date_part, "%d-%m-%y")
            dated_files.append((file_date, file))
        except ValueError:
            # Skip any files that don't match the DD-MM-YY date format prefix
            continue

    if not dated_files:
        return None

    # Sort by datetime (newest date last) and return the path of the latest file
    dated_files.sort(key=lambda x: x[0])
    latest_file = dated_files[-1][1]
    return latest_file


def load_raw_ticker_df() -> pd.DataFrame:
    """Loads the most recent ticker CSV for the configured TICKER_SET into raw_ticker_df."""
    # Handle path resolution safely across .py files and interactive REPLs
    try:
        script_dir = Path(__file__).resolve().parent
    except NameError:
        script_dir = Path.cwd()

    project_root = script_dir.parent if script_dir.name == "src" else script_dir
    ticker_data_dir = project_root / "data" / "ticker_data"

    if not ticker_data_dir.exists():
        raise FileNotFoundError(
            f"Data directory '{ticker_data_dir}' does not exist."
        )

    latest_file_path = get_latest_ticker_file(TICKER_SET, ticker_data_dir)

    if latest_file_path is None:
        raise FileNotFoundError(
            f"No matching files found for TICKER_SET '{TICKER_SET}' in {ticker_data_dir}"
        )

    print(f"Loading most recent ticker file: {latest_file_path.name}")
    raw_ticker_df = pd.read_csv(latest_file_path)
    return raw_ticker_df


def filter_tickers(
    df: pd.DataFrame,
    industries: Optional[Union[str, List[str]]] = None,
    min_price: Optional[float] = None,
    max_price: Optional[float] = None,
    min_market_cap: Optional[float] = None,
    max_market_cap: Optional[float] = None,
) -> List[str]:

    filtered_df = df.copy()

    # Filter by single or multiple industries
    if industries:
        # Convert single string to a list for uniform processing
        if isinstance(industries, str):
            industry_list = [industries]
        else:
            industry_list = industries

        # Option A: Exact Match for any industry in the list
        exact_match = filtered_df["industry"].isin(industry_list)

        # Option B: Partial/Substring Match using Regex OR (|) condition
        # Escape special characters (e.g., handles hyphen/ampersand safely)
        escaped_terms = [re.escape(ind) for ind in industry_list]
        regex_pattern = "|".join(escaped_terms)

        partial_match = filtered_df["industry"].str.contains(
            regex_pattern, case=False, na=False
        )

        # Combine exact and partial matches
        filtered_df = filtered_df[exact_match | partial_match]

    # Apply Price & Market Cap filters...
    if min_price is not None:
        filtered_df = filtered_df[filtered_df["price"] >= min_price]
    if max_price is not None:
        filtered_df = filtered_df[filtered_df["price"] <= max_price]
    if min_market_cap is not None:
        filtered_df = filtered_df[filtered_df["marketCap"] >= min_market_cap]
    if max_market_cap is not None:
        filtered_df = filtered_df[filtered_df["marketCap"] <= max_market_cap]

    return filtered_df["symbol"].dropna().tolist()


# if __name__ == "__main__": !!!!!!!!!!!!!!!!!!


def get_finnhub_api_key(env_path: Optional[Path] = None) -> str:
    """Loads the .env file and retrieves the Finnhub API key."""
    if env_path is None:
        try:
            script_dir = Path(__file__).resolve().parent
        except NameError:
            script_dir = Path.cwd()

        project_root = script_dir.parent if script_dir.name == "src" else script_dir
        env_path = project_root / ".env"

    load_dotenv(dotenv_path=env_path)
    api_key = os.getenv("FINNHUB_API_KEY")

    if not api_key:
        raise ValueError(
            f"FINNHUB_API_KEY not found in environment or file at '{env_path}'."
        )

    return api_key


# ==========================================
# 2. INDIVIDUAL TICKER FETCHERS
# ==========================================

# def fetch_ticker_yahoo_data(symbol: str) -> Dict[str, Any]:
#     """Fetches key metrics, historical price data, financial statements, analyst recommendations/targets, and corporate actions/dividends from Yahoo Finance for a single ticker."""
#     yahoo_data: Dict[str, Any] = {}
#     ticker_obj = yf.Ticker(symbol)

#     def safe_get(key: str, attribute_name: str, is_callable: bool = False, *args, **kwargs):
#         """Helper to execute yfinance property/method calls safely."""
#         try:
#             attr = getattr(ticker_obj, attribute_name)
#             res = attr(*args, **kwargs) if is_callable else attr

#             # Convert DataFrames/Series to dictionaries for easy JSON serialization
#             if isinstance(res, (pd.DataFrame, pd.Series)):
#                 return res.to_dict()
#             return res
#         except Exception as e:
#             return {"error": f"Failed to fetch {key}: {str(e)}"}

#     # 1. Info & Fast Info
#     yahoo_data["info"] = safe_get("info", "info")
#     yahoo_data["fast_info"] = safe_get("fast_info", "fast_info")

#     # 2. Historical Prices (1 Month, Daily)
#     yahoo_data["history"] = safe_get(
#         "history", "history", is_callable=True, period="1mo", interval="1d"
#     )

#     # 3. Financial Statements (Annual & Quarterly)
#     yahoo_data["financials"] = {
#         "income_statement": safe_get("financials", "financials"),
#         "quarterly_income_statement": safe_get("quarterly_financials", "quarterly_financials"),
#         "balance_sheet": safe_get("balance_sheet", "balance_sheet"),
#         "quarterly_balance_sheet": safe_get("quarterly_balance_sheet", "quarterly_balance_sheet"),
#         "cash_flow": safe_get("cashflow", "cashflow"),
#         "quarterly_cash_flow": safe_get("quarterly_cashflow", "quarterly_cashflow"),
#     }

#     # 4. Recommendations & Price Targets
#     yahoo_data["recommendations_and_targets"] = {
#         "recommendations": safe_get("recommendations", "recommendations"),
#         "recommendations_summary": safe_get("recommendations_summary", "recommendations_summary"),
#         "upgrades_downgrades": safe_get("upgrades_downgrades", "upgrades_downgrades"),
#         "analyst_price_targets": safe_get("analyst_price_targets", "analyst_price_targets"),
#     }

#     # 5. Actions & Dividends
#     yahoo_data["actions_and_dividends"] = {
#         "actions": safe_get("actions", "actions"),
#         "dividends": safe_get("dividends", "dividends"),
#         "splits": safe_get("splits", "splits"),
#         "capital_gains": safe_get("capital_gains", "capital_gains"),
#     }

#     return yahoo_data

import pandas as pd
import yfinance as yf
from typing import Dict, Any

def fetch_ticker_yahoo_data(symbol: str) -> Dict[str, Any]:
    """Fetches all key metrics, historical price data, financial statements, ETF fund data,
    options chains, analyst recommendations, and corporate actions from Yahoo Finance."""
    yahoo_data: Dict[str, Any] = {}
    ticker_obj = yf.Ticker(symbol)

    def safe_get(key: str, attribute_name: str, is_callable: bool = False, *args, **kwargs):
        """Helper to execute yfinance property/method calls safely and convert results to dicts."""
        try:
            attr = getattr(ticker_obj, attribute_name)
            res = attr(*args, **kwargs) if is_callable else attr

            # 1. Convert DataFrames and Series to dictionaries
            if isinstance(res, (pd.DataFrame, pd.Series)):
                return res.to_dict()

            # 2. Extract nested objects (like FundsData or FastInfo)
            if hasattr(res, "__dict__") and not isinstance(res, (dict, list, tuple, str, int, float, bool)):
                out = {}
                for prop in dir(res):
                    if not prop.startswith("_"):
                        val = getattr(res, prop)
                        if callable(val):
                            continue
                        if isinstance(val, (pd.DataFrame, pd.Series)):
                            out[prop] = val.to_dict()
                        else:
                            out[prop] = val
                return out

            # 3. Cast custom dict-like structures
            if hasattr(res, "keys") and not isinstance(res, dict):
                try:
                    return dict(res)
                except Exception:
                    pass

            return res
        except Exception as e:
            return {"error": f"Failed to fetch {key}: {str(e)}"}

    # 1. Info, Fast Info, Identifiers & News
    info_res = safe_get("info", "info")

    if isinstance(info_res, dict):
        info_clean = info_res.copy()
        info_clean.pop("companyOfficers", None)  # Removes the key safely if present
        yahoo_data["info"] = info_clean
    else:
        yahoo_data["info"] = info_res
    # yahoo_data["fast_info"] = safe_get("fast_info", "fast_info")
    # yahoo_data["news"] = safe_get("news", "news")


    return yahoo_data

COMPANY_NEWS_LOOKBACK_DAYS = 60
EARNINGS_LOOKBACK_DAYS = 60
EARNINGS_FUTURE_LOOKAHEAD_DAYS = 30

def fetch_ticker_finnhub_data(
    symbol: str, client: finnhub.Client
) -> Dict[str, Any]:
    """Fetches free-tier market metrics, profile, financials, news, insider sentiment, and earnings calendar from Finnhub for a single ticker.

    Each API call is wrapped in an isolated try/except block to gracefully
    handle tier restrictions or missing data.
    """
    finnhub_data: Dict[str, Any] = {}
    now = datetime.now()

    # Dynamic date calculations based on top-level constants
    news_from_date = (now - timedelta(days=COMPANY_NEWS_LOOKBACK_DAYS)).strftime("%Y-%m-%d")
    earnings_from_date = (now - timedelta(days=EARNINGS_LOOKBACK_DAYS)).strftime("%Y-%m-%d")
    earnings_to_date = (now + timedelta(days=EARNINGS_FUTURE_LOOKAHEAD_DAYS)).strftime("%Y-%m-%d")
    today_date = now.strftime("%Y-%m-%d")

    # Helper function to execute API calls safely
    def safe_api_call(call_name: str, func, *args, **kwargs):
        try:
            res = func(*args, **kwargs)
            time.sleep(0.05)  # Small delay to respect rate limits
            return res
        except Exception as e:
            # Captures access denied, tier limits, or missing data without halting
            return {"error": f"Failed to fetch {call_name}: {str(e)}"}

    # 1. Company Profile
    finnhub_data["profile"] = safe_api_call(
        "company_profile2", client.company_profile2, symbol=symbol
    )

    # 2. Real-time Quote
    finnhub_data["quote"] = safe_api_call("quote", client.quote, symbol)

    # 3. Basic Financials & Ratios
    full_financials = safe_api_call(
        "company_basic_financials",
        client.company_basic_financials,
        symbol=symbol,
        metric="all",
    )
    finnhub_data["basic_financials"] = (
    full_financials.get("metric") if isinstance(full_financials, dict) and "metric" in full_financials else full_financials
)

    # 4. Recommendation Trends
    # finnhub_data["recommendation_trends"] = safe_api_call(
    #     "recommendation_trends", client.recommendation_trends, symbol
    # )

    # # 5. Insider Sentiment
    # finnhub_data["insider_sentiment"] = safe_api_call(
    #     "stock_insider_sentiment",
    #     client.stock_insider_sentiment,
    #     symbol=symbol,
    #     _from=news_from_date,
    #     to=today_date,
    # )

    # # 6. Company Earnings (Quarterly EPS surprises)
    # finnhub_data["company_earnings"] = safe_api_call(
    #     "company_earnings", client.company_earnings, symbol
    # )

    # # 7. Company News
    # finnhub_data["company_news"] = safe_api_call(
    #     "company_news",
    #     client.company_news,
    #     symbol=symbol,
    #     _from=news_from_date,
    #     to=today_date,
    # )

    # # 8. Earnings Calendar (Past events + upcoming lookahead)
    # finnhub_data["earnings_calendar"] = safe_api_call(
    #     "earnings_calendar",
    #     client.earnings_calendar,
    #     _from=earnings_from_date,
    #     to=earnings_to_date,
    #     symbol=symbol,
    # )

    return finnhub_data


# ==========================================
# 3. MASTER RETRIEVAL FUNCTION
# ==========================================
def fetch_all_market_data(
    tickers: List[str], env_path: Optional[Path] = None
) -> Dict[str, List[Dict[str, Any]]]:
    """Master function returning a dictionary where each key is a ticker,

    and the value is a list: [yahoo_data, finnhub_data].
    """
    api_key = get_finnhub_api_key(env_path=env_path)
    finnhub_client = finnhub.Client(api_key=api_key)

    market_data: Dict[str, List[Dict[str, Any]]] = {}

    for symbol in tickers:
        print(f"Fetching Yahoo data for {symbol}...")
        yahoo_data = fetch_ticker_yahoo_data(symbol)
        print(f"Fetching Finnhub data for {symbol}...")
        finnhub_data = fetch_ticker_finnhub_data(symbol, finnhub_client)

        # Output format: { "TICKER": [yahoo_data, finnhub_data] }
        market_data[symbol] = [yahoo_data, finnhub_data]

    return market_data

# Main

In [9]:
def main():
    try:
        raw_ticker_df = load_raw_ticker_df()
        print("\n--- raw_ticker_df Loaded Successfully ---")
        print(raw_ticker_df.head())
        filtered_symbols = filter_tickers(
            df=raw_ticker_df,
            industries=INDUSTRIES_FILTER,
            min_price=MIN_PRICE_FILTER,
            max_price=MAX_PRICE_FILTER,
            min_market_cap=MIN_MARKET_CAP_FILTER,
            max_market_cap=MAX_MARKET_CAP_FILTER,
        )
        print(f"Retrieving Yahoo and Finnhub data for: {filtered_symbols}")
        financial_data = fetch_all_market_data(filtered_symbols)
        return financial_data
    except FileNotFoundError as e:
        print(f"Error: {e}")
        return

result = main()

Loading most recent ticker file: 11-08-26_all_ticker_data.csv

--- raw_ticker_df Loaded Successfully ---
  symbol                                 name   price     marketCap  \
0   NVDA      NVIDIA Corporation Common Stock  217.55  5.264710e+12   
1   AAPL              Apple Inc. Common Stock  308.26  4.498802e+12   
2  GOOGL   Alphabet Inc. Class A Common Stock  357.52  4.372470e+12   
3   GOOG  Alphabet Inc. Class C Capital Stock  355.84  4.351923e+12   
4   MSFT   Microsoft Corporation Common Stock  506.06  3.757772e+12   

      volume    industry  
0  115849078  Technology  
1   44812784  Technology  
2   18991463  Technology  
3   12789519  Technology  
4   31206408  Technology  
Retrieving Yahoo and Finnhub data for: ['AVGO', 'AMD', 'DELL', 'WDC', 'SPGI', 'MCO', 'SNPS', 'MSI', 'BRKRP', 'ROK', 'WAT', 'ROP', 'MDB', 'RL', 'RS', 'LII', 'DY', 'VMI', 'PLPC']
Fetching Yahoo data for AVGO...
Fetching Finnhub data for AVGO...
Fetching Yahoo data for AMD...
Fetching Finnhub data for AMD...

# Output MSFT Data

In [10]:
import json

def sanitize_for_json(obj: Any) -> Any:
    """Recursively converts DataFrames, Series, Timestamps, FastInfo, custom objects, and non-string dict keys into JSON-serializable types."""
    if isinstance(obj, (pd.DataFrame, pd.Series)):
        return sanitize_for_json(obj.to_dict())

    if hasattr(obj, "keys") and not isinstance(obj, dict):
        try:
            return sanitize_for_json({k: obj[k] for k in obj.keys()})
        except Exception:
            return str(obj)

    if isinstance(obj, dict):
        sanitized_dict = {}
        for key, value in obj.items():
            if isinstance(key, (pd.Timestamp, pd.Timedelta, datetime)):
                str_key = key.isoformat()
            elif not isinstance(key, (str, int, float, bool, type(None))):
                str_key = str(key)
            else:
                str_key = key

            sanitized_dict[str_key] = sanitize_for_json(value)
        return sanitized_dict

    if isinstance(obj, (list, tuple, set)):
        return [sanitize_for_json(item) for item in obj]

    if isinstance(obj, (pd.Timestamp, pd.Timedelta, datetime)):
        return obj.isoformat()

    if pd.isna(obj):
        return None

    if not isinstance(obj, (str, int, float, bool, type(None))):
        if hasattr(obj, "__dict__"):
            return sanitize_for_json(obj.__dict__)
        return str(obj)

    return obj


def export_ticker_files(result: Dict[str, List[Dict[str, Any]]], symbol: str) -> None:
    """Exports Yahoo and Finnhub data for a given ticker into separate formatted JSON files."""
    ticker_data = result.get(symbol)

    if not ticker_data or len(ticker_data) < 2:
        print(f"Data for '{symbol}' not found or improperly formatted in result dictionary.")
        return

    yahoo_clean = sanitize_for_json(ticker_data[0])
    finnhub_clean = sanitize_for_json(ticker_data[1])

    yahoo_filename = f"{symbol.lower()}_yahoo_data.json"
    finnhub_filename = f"{symbol.lower()}_finnhub_data.json"

    with open(yahoo_filename, "w", encoding="utf-8") as f:
        json.dump(yahoo_clean, f, indent=4, ensure_ascii=False)

    with open(finnhub_filename, "w", encoding="utf-8") as f:
        json.dump(finnhub_clean, f, indent=4, ensure_ascii=False)

    print(f"Successfully exported '{yahoo_filename}' and '{finnhub_filename}'!")
    
export_ticker_files(result, "AVGO")

Successfully exported 'avgo_yahoo_data.json' and 'avgo_finnhub_data.json'!
